# 5-Minute Quickstart Guide to DuckPD

Welcome to the **DuckPD Quickstart Guide**! This notebook demonstrates how you can run fast, scalable **pandas-like data science workflows directly on top of DuckDB** without managing separate database clusters, servers, or proprietary accounts.

### What is DuckPD?
**DuckPD** brings a lazy, relational pandas-shaped API to the Python ecosystem with DuckDB as the high-performance analytical execution backend. You get the intuitive, familiar DataFrame and Series APIs you love, while execution benefits from:
- **Predicate and projection pushdown**
- **Vectorized multithreaded SQL execution**
- **Constant-memory footprints on large files**
- **Zero silent full-frame materialization**

## Step 1: Import DuckPD

To get started, simply import `duckpd as pd`. Unlike other systems, DuckPD requires **no API keys, no cloud signups, and no cloud warehouse configurations**.

In [1]:
import duckpd as pd

print(f"DuckPD version: {pd.__version__}")

DuckPD version: 0.0.1.dev0


## Step 2: Establish a Session

Connect to an embedded DuckDB session. You can customize memory limits, thread pools, and temporary spill directories easily.

In [2]:
session = pd.connect(memory_limit="512MB", threads=4)
print(f"Connected to DuckDB backend. Active executions: {session.execution_count}")

Connected to DuckDB backend. Active executions: 0


## Step 3: Load Data Directly via DuckDB

We load the classic **Goodreads Books dataset** directly from the web using DuckDB's automatic CSV scanner. Notice that DuckPD creates a **lazy plan**—no data rows are loaded into Python memory yet!

In [3]:
books_url = (
    "https://raw.githubusercontent.com/ponder-org/ponder-datasets/main/books.csv"
)

# Scan remote CSV directly into a lazy DuckPD DataFrame
df = session.sql(f"SELECT * FROM read_csv_auto('{books_url}')")

print("Lazy DataFrame:")
print(df)
print(f"\nColumns: {df.columns}")
print(f"Total element count (size): {df.size}")

Lazy DataFrame:
DuckPD DataFrame
Columns: ['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', 'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']
Plan: ScanPlan

Columns: ('bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', 'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher')
Total element count (size): 133476


## Step 4: Preview Data (Bounded Eager Head)

Calling `head()` executes a bounded preview with `LIMIT` pushed down into the database, returning only the requested rows into Python.

In [4]:
df[["bookID", "title", "authors", "average_rating", "num_pages", "ratings_count"]].head(
    5
)

,bookID,title,authors,average_rating,num_pages,ratings_count
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,652,2095690
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,870,2153167
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,352,6333
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,435,2339585
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,2690,41428


## Step 5: Column Reductions & Summary Statistics

Calculate summary statistics on the numerical columns. Each reduction computes in a single, fast SQL query in DuckDB.

In [5]:
num_cols = ["average_rating", "num_pages", "ratings_count", "text_reviews_count"]

print("--- Non-null counts ---")
print(df[num_cols].count())

print("\n--- Means ---")
print(df[num_cols].mean())

print("\n--- Minima ---")
print(df[num_cols].min())

print("\n--- Maxima ---")
print(df[num_cols].max())

--- Non-null counts ---
average_rating        11123
num_pages             11123
ratings_count         11123
text_reviews_count    11123
dtype: int64

--- Means ---
average_rating            3.934075
num_pages               336.405556
ratings_count         17942.848063
text_reviews_count      542.048099
dtype: float64

--- Minima ---
average_rating        0.0
num_pages             0.0
ratings_count         0.0
text_reviews_count    0.0
dtype: float64

--- Maxima ---
average_rating              5.0
num_pages                6576.0
ratings_count         4597666.0
text_reviews_count      94265.0
dtype: float64


## Step 6: Vectorized String Operations (`.str` accessor)

Use pandas-compatible string methods executed natively in DuckDB.

In [6]:
# Filter Harry Potter books and standardize author names
hp_books = (
    df[df["title"].str.contains("Harry Potter")]
    .assign(
        author_upper=lambda f: f["authors"].str.upper(),
        title_len=lambda f: f["title"].str.len(),
    )[["bookID", "title", "author_upper", "average_rating", "num_pages"]]
    .sort_values("average_rating", ascending=False)
)

hp_books.head(5)

,bookID,title,author_upper,average_rating,num_pages
0,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. ROWLING/MARY GRANDPRÉ,4.78,2690
1,10,Harry Potter Collection (Harry Potter #1-6),J.K. ROWLING,4.73,3342
2,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. ROWLING/MARY GRANDPRÉ,4.57,652
3,2005,Harry Potter and the Half-Blood Prince (Harry ...,J.K. ROWLING,4.57,768
4,15872,Harry Potter y el misterio del príncipe (Harry...,J.K. ROWLING/GEMMA ROVIRA ORTEGA,4.57,602


## Step 7: GroupBy & Aggregations

Perform analytical grouping across categories with named aggregations.

In [7]:
language_summary = (
    df.groupby("language_code", as_index=False)
    .agg(
        total_books=("bookID", "size"),
        avg_rating=("average_rating", "mean"),
        total_pages=("num_pages", "sum"),
        max_reviews=("text_reviews_count", "max"),
    )
    .sort_values("total_books", ascending=False)
)

print("Top 10 Languages by Book Count:")
language_summary.head(10)

Top 10 Languages by Book Count:


,language_code,total_books,avg_rating,total_pages,max_reviews
0,eng,8908,3.934062,3000191,94265
1,en-US,1408,3.914659,465259,8119
2,spa,218,3.929312,79743,3647
3,en-GB,214,3.923411,67519,3838
4,fre,144,3.971528,48816,2088
5,ger,99,3.950101,38806,343
6,jpn,46,4.268696,8780,38
7,mul,19,4.126316,8308,160
8,zho,14,4.456429,4506,4
9,grc,11,3.707273,4547,10


## Step 8: Inspecting Query Execution (`explain`)

DuckPD lets you inspect the full query plan, generated SQL, and DuckDB physical pipeline at any time!

In [8]:
print(language_summary.explain())

DuckPD logical plan:
SortPlan(input=AggregatePlan(input=ScanPlan(source=SqlSource(query="SELECT * FROM read_csv_auto('https://raw.githubusercontent.com/ponder-org/ponder-datasets/main/books.csv')"), metadata=FrameMetadata(columns=(Column(id=ColumnId(value=UUID('2646e5cb-8e92-435c-90dd-33c9a8e3926c')), label='bookID', duckdb_type='BIGINT', hidden=False), Column(id=ColumnId(value=UUID('d45e9634-b23b-4e27-97a9-c43c7a5e87dd')), label='title', duckdb_type='VARCHAR', hidden=False), Column(id=ColumnId(value=UUID('487912a7-7750-4dc4-93d8-8b0e50057ba0')), label='authors', duckdb_type='VARCHAR', hidden=False), Column(id=ColumnId(value=UUID('279cfe4a-01c8-4aab-8fec-9ad53712512f')), label='average_rating', duckdb_type='DOUBLE', hidden=False), Column(id=ColumnId(value=UUID('5120a314-d219-462d-8de6-a60a61185260')), label='isbn', duckdb_type='VARCHAR', hidden=False), Column(id=ColumnId(value=UUID('8970e12a-4240-4fde-bdc7-5c7360f7d9a0')), label='isbn13', duckdb_type='VARCHAR', hidden=False), Column(id

## Step 9: Exporting Results Directly

When you're ready, you can collect the result back into a standard pandas DataFrame or write it directly into Parquet without materializing in Python heap.

In [9]:
# Export to pandas
pandas_df = language_summary.head(5)
print(type(pandas_df))
display(pandas_df)

# Direct write to Parquet
language_summary.write_parquet("language_summary.parquet", overwrite=True)
print("\nSuccessfully exported language_summary.parquet directly from DuckDB!")

<class 'pandas.DataFrame'>


,language_code,total_books,avg_rating,total_pages,max_reviews
0,eng,8908,3.934062,3000191,94265
1,en-US,1408,3.914659,465259,8119
2,spa,218,3.929312,79743,3647
3,en-GB,214,3.923411,67519,3838
4,fre,144,3.971528,48816,2088



Successfully exported language_summary.parquet directly from DuckDB!
